In [ ]:
%sql
-- 1. Create Metadata Schema in Unity Catalog
CREATE SCHEMA IF NOT EXISTS spark_training.metadata_schema;

In [ ]:
%sql
-- 2. Create Metadata Control Table with PK, Batch & Tracking Attributes
CREATE TABLE IF NOT EXISTS spark_training.metadata_schema.metadata_table(
    table_id INT PRIMARY KEY,
    source_name VARCHAR(50),
    source_table_name VARCHAR(100),
    source_schema_name VARCHAR(100),
    source_database_name VARCHAR(100),
    target_name VARCHAR(50),
    target_table_name VARCHAR(100),
    target_schema_name VARCHAR(100),
    target_database_name VARCHAR(100),
    merge_key VARCHAR(100),
    last_load_date_column VARCHAR(100),
    last_load_date TIMESTAMP,
    query STRING,
    load_type VARCHAR(100),
    batch_id INT,
    is_active VARCHAR(100),
    created_on DATE,
    created_by VARCHAR(100),
    updated_on DATE,
    updated_by VARCHAR(100),
    last_run_status VARCHAR(100)
);

In [ ]:
%sql
-- 3. Insert or Update Pipeline Control Metadata for Bronze Layer (Idempotent MERGE with batch_id)
-- Bronze ID Range: 101-199 | Silver ID Range: 201-299 | Gold ID Range: 301-399
MERGE INTO spark_training.metadata_schema.metadata_table AS target
USING (
    -- Batch 1: Fast Lookup Dimensions (101-104)
    SELECT 101 AS table_id, 'sql server' AS source_name, 'countries' AS source_table_name, 'dbo' AS source_schema_name, 'wanderbricks' AS source_database_name, 'bronze' AS target_name, 'countries' AS target_table_name, 'bronze' AS target_schema_name, 'spark_training' AS target_database_name, 'country' AS merge_key, CAST(NULL AS STRING) AS last_load_date_column, CAST(NULL AS TIMESTAMP) AS last_load_date, 'SELECT country, country_code, continent FROM countries' AS query, 'incremental' AS load_type, 1 AS batch_id, 'true' AS is_active, CURRENT_DATE() AS created_on, 'sham' AS created_by, CURRENT_DATE() AS updated_on, 'sham' AS updated_by, CAST(NULL AS STRING) AS last_run_status
    UNION ALL SELECT 102, 'sql server', 'destinations', 'dbo', 'wanderbricks', 'bronze', 'destinations', 'bronze', 'spark_training', 'destination_id', NULL, NULL, 'SELECT destination_id, destination, country, state_or_province, state_or_province_code, description FROM destinations', 'incremental', 1, 'true', CURRENT_DATE(), 'sham', CURRENT_DATE(), 'sham', NULL
    UNION ALL SELECT 103, 'sql server', 'amenities', 'dbo', 'wanderbricks', 'bronze', 'amenities', 'bronze', 'spark_training', 'amenity_id', NULL, NULL, 'SELECT amenity_id, name, category, icon FROM amenities', 'incremental', 1, 'true', CURRENT_DATE(), 'sham', CURRENT_DATE(), 'sham', NULL
    UNION ALL SELECT 104, 'sql server', 'hosts', 'dbo', 'wanderbricks', 'bronze', 'hosts', 'bronze', 'spark_training', 'host_id', 'joined_at', NULL, 'SELECT host_id, name, email, phone, is_verified, is_active, rating, country, joined_at FROM hosts', 'incremental', 1, 'true', CURRENT_DATE(), 'sham', CURRENT_DATE(), 'sham', NULL
    -- Batch 2: Core Entities & Relations (105-107)
    UNION ALL SELECT 105, 'sql server', 'users', 'dbo', 'wanderbricks', 'bronze', 'users', 'bronze', 'spark_training', 'user_id', 'created_at', NULL, 'SELECT user_id, email, name, country, user_type, created_at, is_business, company_name FROM users', 'incremental', 2, 'true', CURRENT_DATE(), 'sham', CURRENT_DATE(), 'sham', NULL
    UNION ALL SELECT 106, 'sql server', 'properties', 'dbo', 'wanderbricks', 'bronze', 'properties', 'bronze', 'spark_training', 'property_id', 'created_at', NULL, 'SELECT property_id, host_id, destination_id, title, description, base_price, property_type, max_guests, bedrooms, bathrooms, property_latitude, property_longitude, created_at FROM properties', 'incremental', 2, 'true', CURRENT_DATE(), 'sham', CURRENT_DATE(), 'sham', NULL
    UNION ALL SELECT 107, 'sql server', 'property_amenities', 'dbo', 'wanderbricks', 'bronze', 'property_amenities', 'bronze', 'spark_training', 'property_id, amenity_id', NULL, NULL, 'SELECT property_id, amenity_id FROM property_amenities', 'incremental', 2, 'true', CURRENT_DATE(), 'sham', CURRENT_DATE(), 'sham', NULL
    -- Batch 3: Transactions & Reviews (108-110)
    UNION ALL SELECT 108, 'sql server', 'bookings', 'dbo', 'wanderbricks', 'bronze', 'bookings', 'bronze', 'spark_training', 'booking_id', 'updated_at', NULL, 'SELECT booking_id, user_id, property_id, check_in, check_out, guests_count, total_amount, status, created_at, updated_at FROM bookings', 'incremental', 3, 'true', CURRENT_DATE(), 'sham', CURRENT_DATE(), 'sham', NULL
    UNION ALL SELECT 109, 'sql server', 'payments', 'dbo', 'wanderbricks', 'bronze', 'payments', 'bronze', 'spark_training', 'payment_id', 'payment_date', NULL, 'SELECT payment_id, booking_id, amount, payment_method, status, payment_date FROM payments', 'incremental', 3, 'true', CURRENT_DATE(), 'sham', CURRENT_DATE(), 'sham', NULL
    UNION ALL SELECT 110, 'sql server', 'reviews', 'dbo', 'wanderbricks', 'bronze', 'reviews', 'bronze', 'spark_training', 'review_id', 'updated_at', NULL, 'SELECT review_id, booking_id, user_id, property_id, rating, comment, is_deleted, created_at, updated_at FROM reviews', 'incremental', 3, 'true', CURRENT_DATE(), 'sham', CURRENT_DATE(), 'sham', NULL
    -- Batch 4: High-Volume Event Telemetry (111-112)
    UNION ALL SELECT 111, 'sql server', 'page_views', 'dbo', 'wanderbricks', 'bronze', 'page_views', 'bronze', 'spark_training', 'view_id', 'timestamp', NULL, 'SELECT view_id, user_id, property_id, device_type, page_url, referrer, timestamp FROM page_views', 'incremental', 4, 'true', CURRENT_DATE(), 'sham', CURRENT_DATE(), 'sham', NULL
    UNION ALL SELECT 112, 'sql server', 'clickstream', 'dbo', 'wanderbricks', 'bronze', 'clickstream', 'bronze', 'spark_training', 'user_id, property_id, timestamp', 'timestamp', NULL, 'SELECT event, metadata, property_id, timestamp, user_id FROM clickstream', 'incremental', 4, 'true', CURRENT_DATE(), 'sham', CURRENT_DATE(), 'sham', NULL
) AS source
ON target.table_id = source.table_id
WHEN MATCHED THEN UPDATE SET
    target.source_name = source.source_name,
    target.source_table_name = source.source_table_name,
    target.source_schema_name = source.source_schema_name,
    target.source_database_name = source.source_database_name,
    target.target_name = source.target_name,
    target.target_table_name = source.target_table_name,
    target.target_schema_name = source.target_schema_name,
    target.target_database_name = source.target_database_name,
    target.merge_key = source.merge_key,
    target.last_load_date_column = source.last_load_date_column,
    target.query = source.query,
    target.load_type = source.load_type,
    target.batch_id = source.batch_id,
    target.is_active = source.is_active,
    target.updated_on = source.updated_on,
    target.updated_by = source.updated_by
WHEN NOT MATCHED THEN INSERT (
    table_id, source_name, source_table_name, source_schema_name, source_database_name,
    target_name, target_table_name, target_schema_name, target_database_name,
    merge_key, last_load_date_column, last_load_date, query, load_type, batch_id, is_active,
    created_on, created_by, updated_on, updated_by, last_run_status
) VALUES (
    source.table_id, source.source_name, source.source_table_name, source.source_schema_name, source.source_database_name,
    source.target_name, source.target_table_name, source.target_schema_name, source.target_database_name,
    source.merge_key, source.last_load_date_column, source.last_load_date, source.query, source.load_type, source.batch_id, source.is_active,
    source.created_on, source.created_by, source.updated_on, source.updated_by, source.last_run_status
);

In [ ]:
%sql
-- 4. Verify Configured Control Records
SELECT 
    table_id,
    source_name,
    source_table_name,
    target_name,
    target_table_name,
    merge_key,
    last_load_date_column,
    load_type,
    is_active,
    created_by,
    query
FROM spark_training.metadata_schema.metadata_table
ORDER BY table_id;